Oracle AI Data Platform v1.0

# MongoDB Ingestion: Full and Incremental

Lands one MongoDB collection in a Delta table in the catalog and schema named in `mongodb_ingest.yaml`, using the `aidp-connector-mongodb` package.

- **Incremental (default):** reads only documents whose `watermark_field` changed since the last run, or the whole collection when it is `null`, and merges them on `_id`.
- **Full:** reloads the whole collection and overwrites the table.

**Before you run**

1. Install the MongoDB Spark Connector jar and the `aidp-connector-mongodb` wheel as cluster libraries, then restart the cluster.
2. Store a read-only user's connection string in the AIDP Credential Store and set `mongodb.credential_name` in the config.
3. Add the cluster's outbound IP to the Atlas project's IP access list.
4. Copy `mongodb_ingest.sample.yaml` to `mongodb_ingest.yaml` in your workspace and edit the collection and the target.

See the package `README.md` for details and known limits.

## 1. Install the connector

Skip this cell if the wheel is already installed as a cluster library. Otherwise upload `aidp_connector_mongodb-<version>-py3-none-any.whl` to your workspace, then uncomment the line below and set its path. It also installs `pyyaml`.

In [ ]:
# %pip install /Workspace/<path-to>/aidp_connector_mongodb-0.1.0-py3-none-any.whl

In [ ]:
from aidp_connector_mongodb import __version__, format_summary, load_config, read_uri, run

print("aidp-connector-mongodb", __version__)

## 2. Load the configuration

In [ ]:
# Your copy of mongodb_ingest.sample.yaml.
CONFIG_PATH = "/Workspace/<path-to>/mongodb_ingest.yaml"

config = load_config(CONFIG_PATH)

print("collection:", f"{config.mongodb.database}.{config.mongodb.collection}")
print("target    :", config.target.qualified_table)
print("watermark :", config.sync.watermark_field or "(none: full re-read every run)")

## 3. Optional per-run override

When this notebook runs as a job, a job parameter `MODE`, `incremental` or `full`, sets the mode for that run only. Interactive runs are incremental.

In [ ]:
_MISSING = "\x00__NO_PARAMETER__"


def job_parameter(name):
    """Read a job parameter, or None. Names are case-sensitive on the platform,
    so try the usual spellings: whoever registers the job may write 'mode', not 'MODE'."""
    for candidate in dict.fromkeys((name, name.upper(), name.lower())):
        try:
            # oidlUtils is provided by the AIDP runtime - do not import it.
            value = oidlUtils.parameters.getParameter(candidate, _MISSING)  # noqa: F821
        except NameError:
            return None  # not running on AIDP
        if not (value == _MISSING or value is None or str(value).strip() == ""):
            return str(value).strip()
    return None


MODE = job_parameter("MODE")
print("mode override:", MODE or "(none)")

## 4. Read the connection URI

In [ ]:
try:
    get_secret = aidputils.secrets.get  # noqa: F821 - provided by the AIDP runtime, do not import it
except NameError:
    get_secret = None  # not on AIDP: read the URI from the environment variable instead

uri = read_uri(config.mongodb, get_secret)
print("URI read (not shown)")

## 5. Run the sync

In [ ]:
summary = run(spark, config, uri, MODE)  # noqa: F821 - spark is provided by the notebook

print(format_summary(summary))

## 6. Look at the result

In [ ]:
table = config.target.qualified_table

spark.table(table).show(20)  # noqa: F821 - spark is provided by the notebook
# No duplicates: both counts are equal.
spark.sql(f"SELECT count(*) AS row_count, count(DISTINCT _id) AS distinct_ids FROM {table}").show()  # noqa: F821

## Connector Options

Set in `mongodb_ingest.yaml`.

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `mongodb.credential_name` | Credential Store entry | Y | Entry holding the connection string. |
| `mongodb.credential_key` | Key name | N | Key inside that entry. Default `uri`. |
| `mongodb.database` | Database name | Y | Database to read from. |
| `mongodb.collection` | Collection name | Y | Collection to read from. |
| `target.catalog` | Existing catalog | Y | Target catalog. |
| `target.schema` | Schema name | Y | Target schema. Created if missing. |
| `target.table` | Table name | Y | Target table. |
| `sync.watermark_field` | Date field, or `null` | N | Field used to read only changed documents. `null` re-reads the whole collection. |
| `sync.fields` | List of field names | N | Fields to load. `[]` loads every field. |
| `sync.string_fields` | List of field names | N | Fields read as text because their type differs between documents. |